#  BiLSTM 텍스트 분류

## LSTM vs Vanilla RNN

**Vanilla RNN** — gradient 소실/폭발 문제
$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

**LSTM** — 4개 게이트로 정보 흐름 제어
$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \quad \text{(Forget gate)}$$
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i) \quad \text{(Input gate)}$$
$$g_t = \tanh(W_g [h_{t-1}, x_t] + b_g) \quad \text{(Cell candidate)}$$
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o) \quad \text{(Output gate)}$$
$$c_t = f_t \odot c_{t-1} + i_t \odot g_t, \quad h_t = o_t \odot \tanh(c_t)$$

## Bidirectional LSTM

```
→ 순방향: x1 → x2 → x3 → ... → xT
← 역방향: x1 ← x2 ← x3 ← ... ← xT
최종: h_t = [h_t_fwd ; h_t_bwd]  ← 앞뒤 문맥 모두 반영
```

## 풀링 전략 비교

| 전략 | 방법 | 특징 |
|------|------|------|
| **last** | 마지막 hidden state | 빠르지만 앞부분 정보 손실 가능 |
| **max** | 전체 시퀀스 Max Pool | 가장 강한 특징 추출 |
| **attention** | 가중합 (Bahdanau) | 중요 단어에 집중, 해석 가능 |

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random

torch.manual_seed(42); random.seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


## 1️⃣ 데이터셋 (주제 분류 — 4클래스)

In [2]:
TOPIC_DATA = [
    # 스포츠 (0)
    ('the team scored an amazing goal in extra time',           0),
    ('basketball player made incredible three point shot',      0),
    ('marathon runner finished the race in record time',        0),
    ('soccer championship was won by the home team',            0),
    ('tennis player aced the serve in the final set',           0),
    ('swimming champion broke the world record again',          0),
    ('baseball pitcher threw a perfect game last night',        0),
    ('the athlete trained hard for the olympic games',          0),
    ('football coach announced new strategy for season',        0),
    ('cycling tour champion crossed the finish line first',     0),
    # 기술 (1)
    ('the new artificial intelligence model is amazing',        1),
    ('software engineers developed new machine learning algorithm', 1),
    ('deep neural networks achieved state of the art results',  1),
    ('technology company released faster computer processor',   1),
    ('smartphone apps use natural language processing features',1),
    ('cloud computing services enable scalable applications',   1),
    ('researchers published study on transformer architecture', 1),
    ('robot arm performs precise manufacturing operations',     1),
    ('quantum computing breakthrough announced by scientists',  1),
    ('open source library accelerates deep learning training',  1),
    # 음식 (2)
    ('the chef prepared delicious pasta with fresh tomatoes',   2),
    ('baking sourdough bread requires patience and practice',   2),
    ('restaurant serves amazing sushi with fresh ingredients',  2),
    ('homemade chocolate cake tastes better than store bought', 2),
    ('grilling steak over charcoal gives best flavor',          2),
    ('vegetable curry with coconut milk is very healthy',       2),
    ('french bakery offers wonderful croissants every morning', 2),
    ('slow cooked beef stew warms you on cold winter days',     2),
    ('fresh salad with olive oil and lemon is refreshing',      2),
    ('spicy kimchi is a traditional korean fermented dish',     2),
    # 여행 (3)
    ('visiting paris in spring is absolutely wonderful experience', 3),
    ('backpacking through southeast asia was life changing trip',   3),
    ('the mountain hike offered breathtaking views of valley',      3),
    ('tropical beach vacation relaxed mind and rejuvenated body',   3),
    ('ancient temples in kyoto reflect centuries of japanese culture', 3),
    ('road trip through national parks revealed natural wonders',   3),
    ('cruise ship traveled through the caribbean sea islands',      3),
    ('camping under the stars in desert was unforgettable night',   3),
    ('historic city tour revealed fascinating architectural details',3),
    ('safari in africa provided close encounter with wildlife',     3),
]

LABEL_NAMES = {0: '스포츠', 1: '기술', 2: '음식', 3: '여행'}

class Vocab:
    def __init__(self):
        self.w2i = {'<PAD>': 0, '<UNK>': 1}
    def build(self, texts):
        for text in texts:
            for w in text.lower().split():
                if w not in self.w2i:
                    self.w2i[w] = len(self.w2i)
        print(f'[Vocab] 크기: {len(self.w2i)}')
    def encode(self, text, max_len):
        ids = [self.w2i.get(w, 1) for w in text.lower().split()]
        length = min(len(ids), max_len)
        return (ids + [0]*max_len)[:max_len], length
    def __len__(self): return len(self.w2i)


class TopicDataset(Dataset):
    def __init__(self, data, vocab, max_len=20):
        self.samples = []
        for text, label in data:
            ids, length = vocab.encode(text, max_len)
            self.samples.append((
                torch.tensor(ids,    dtype=torch.long),
                torch.tensor(length, dtype=torch.long),
                torch.tensor(label,  dtype=torch.long),
            ))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


random.shuffle(TOPIC_DATA)
split = int(len(TOPIC_DATA) * 0.8)
train_data, test_data = TOPIC_DATA[:split], TOPIC_DATA[split:]

vocab = Vocab()
vocab.build([t for t, _ in TOPIC_DATA])

MAX_LEN = 20
train_loader = DataLoader(TopicDataset(train_data, vocab, MAX_LEN), batch_size=8, shuffle=True)
test_loader  = DataLoader(TopicDataset(test_data,  vocab, MAX_LEN), batch_size=8)
print(f'Train: {len(train_data)}  /  Test: {len(test_data)}')

[Vocab] 크기: 247
Train: 32  /  Test: 8


## 2️⃣ Attention 모듈 (Bahdanau Additive Attention)

In [3]:
class AdditiveAttention(nn.Module):
    """
    Bahdanau-style Attention
    score(h_t) = v · tanh(W · h_t)
    α_t        = softmax(score)
    context    = Σ α_t · h_t
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden_states, mask=None):
        scores = self.v(torch.tanh(self.W(hidden_states))).squeeze(-1)  # (B, T)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)                          # (B, T)
        context = (weights.unsqueeze(-1) * hidden_states).sum(dim=1)    # (B, H)
        return context, weights

## 3️⃣ BiLSTM 분류기

In [4]:
class BiLSTMClassifier(nn.Module):
    """
    풀링 전략:
      'last'      → 순/역방향 마지막 hidden 연결
      'max'       → 전체 시퀀스 Max Pooling
      'attention' → Bahdanau Attention 가중합
    """
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64,
                 num_layers=2, num_classes=4, dropout=0.3,
                 pooling='attention', pad_idx=0):
        super().__init__()
        assert pooling in ('last', 'max', 'attention')
        self.pooling = pooling

        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embed_drop = nn.Dropout(dropout)

        self.bilstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        lstm_out = hidden_dim * 2  # 양방향 → 2배
        if pooling == 'attention':
            self.attention = AdditiveAttention(lstm_out)

        self.classifier = nn.Sequential(
            nn.LayerNorm(lstm_out),
            nn.Dropout(dropout),
            nn.Linear(lstm_out, lstm_out // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_out // 2, num_classes),
        )

    def forward(self, x, lengths=None):
        emb = self.embed_drop(self.embedding(x))        # (B, L, E)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
            out_packed, (h_n, _) = self.bilstm(packed)
            out, _ = nn.utils.rnn.pad_packed_sequence(
                out_packed, batch_first=True, total_length=x.size(1))
        else:
            out, (h_n, _) = self.bilstm(emb)

        if self.pooling == 'last':
            rep = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (B, 2H)
        elif self.pooling == 'max':
            rep = out.max(dim=1).values
        elif self.pooling == 'attention':
            mask = (x != 0)
            rep, self.attn_weights = self.attention(out, mask)

        return self.classifier(rep)

## 4️⃣ 학습 함수 & 기본 모델 학습

In [5]:
def run_experiment(model_cfg, train_loader, test_loader, epochs=50, verbose=True):
    model = BiLSTMClassifier(vocab_size=len(vocab), **model_cfg).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_acc  = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        for x, lengths, y in train_loader:
            x, lengths, y = x.to(device), lengths.to(device), y.to(device)
            optimizer.zero_grad()
            criterion(model(x, lengths), y).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, lengths, y in test_loader:
                x, lengths, y = x.to(device), lengths.to(device), y.to(device)
                correct += (model(x, lengths).argmax(1) == y).sum().item()
                total   += y.size(0)
        acc = correct / total
        if acc > best_acc:
            best_acc = acc

        if verbose and epoch % 10 == 0:
            print(f'  Epoch {epoch:3d} | Acc: {acc*100:.1f}%')

    return best_acc, model


print('=' * 50)
print('[기본 BiLSTM + Attention 학습]')
print('=' * 50)
base_cfg = dict(hidden_dim=64, num_layers=2, dropout=0.3,
                pooling='attention', embed_dim=64, num_classes=4)
best_acc, best_model = run_experiment(base_cfg, train_loader, test_loader, epochs=50)
print(f'\n최고 Test Accuracy: {best_acc*100:.1f}%')

[기본 BiLSTM + Attention 학습]
  Epoch  10 | Acc: 25.0%
  Epoch  20 | Acc: 37.5%
  Epoch  30 | Acc: 37.5%
  Epoch  40 | Acc: 37.5%
  Epoch  50 | Acc: 37.5%

최고 Test Accuracy: 37.5%


## 5️⃣ 혼동 행렬 & 클래스별 성능

In [6]:
@torch.no_grad()
def confusion_matrix_report(model, loader, num_classes=4):
    model.eval()
    all_pred, all_true = [], []
    for x, lengths, y in loader:
        x, lengths = x.to(device), lengths.to(device)
        all_pred.extend(model(x, lengths).argmax(1).cpu().tolist())
        all_true.extend(y.tolist())

    cm = [[0]*num_classes for _ in range(num_classes)]
    for t, p in zip(all_true, all_pred):
        cm[t][p] += 1

    print('\n[혼동 행렬]')
    print('         ' + '  '.join(f'{LABEL_NAMES[i]:>6}' for i in range(num_classes)))
    for i, row in enumerate(cm):
        print(f'{LABEL_NAMES[i]:>6} | ' + '  '.join(f'{v:>6}' for v in row))

    print('\n[클래스별 성능]')
    print(f"  {'클래스':>6} | {'Precision':>10} | {'Recall':>8} | {'F1':>8}")
    print(f"  {'-'*42}")
    for i in range(num_classes):
        tp = cm[i][i]
        fp = sum(cm[j][i] for j in range(num_classes)) - tp
        fn = sum(cm[i][j] for j in range(num_classes)) - tp
        p  = tp/(tp+fp) if tp+fp>0 else 0
        r  = tp/(tp+fn) if tp+fn>0 else 0
        f1 = 2*p*r/(p+r) if p+r>0 else 0
        print(f"  {LABEL_NAMES[i]:>6} | {p:>10.3f} | {r:>8.3f} | {f1:>8.3f}")

confusion_matrix_report(best_model, test_loader)


[혼동 행렬]
            스포츠      기술      음식      여행
   스포츠 |      2       0       0       3
    기술 |      0       1       0       2
    음식 |      0       0       0       0
    여행 |      0       0       0       0

[클래스별 성능]
     클래스 |  Precision |   Recall |       F1
  ------------------------------------------
     스포츠 |      1.000 |    0.400 |    0.571
      기술 |      1.000 |    0.333 |    0.500
      음식 |      0.000 |    0.000 |    0.000
      여행 |      0.000 |    0.000 |    0.000


## 6️⃣ Attention 가중치 시각화

In [7]:
@torch.no_grad()
def visualize_attention(model, vocab, sentence, max_len=20):
    if model.pooling != 'attention':
        print('Attention pooling 모델만 시각화 가능')
        return
    model.eval()
    words = sentence.lower().split()
    ids, length = vocab.encode(sentence, max_len)
    x       = torch.tensor([ids],    device=device)
    lengths = torch.tensor([length], device=device)
    logits  = model(x, lengths)
    probs   = torch.softmax(logits, dim=1)[0]
    weights = model.attn_weights[0, :length].cpu().numpy()

    pred = LABEL_NAMES[probs.argmax().item()]
    print(f"\n문장: '{sentence}'  →  예측: {pred}")
    print(f"  {'단어':>18} | {'가중치':>7} | 시각화")
    print(f"  {'-'*50}")
    for w, a in sorted(zip(words, weights), key=lambda x: -x[1]):
        bar = '▓' * int(a * 30)
        print(f"  {w:>18} | {a:.4f}  | {bar}")


vis_sents = [
    'the player scored amazing goal in championship game',
    'neural network model achieved best performance results',
    'chef cooked delicious pasta with fresh ingredients',
    'traveling through mountains offered breathtaking views',
]
for sent in vis_sents:
    visualize_attention(best_model, vocab, sent)


문장: 'the player scored amazing goal in championship game'  →  예측: 스포츠
                  단어 |     가중치 | 시각화
  --------------------------------------------------
                 the | 0.1386  | ▓▓▓▓
              scored | 0.1282  | ▓▓▓
        championship | 0.1267  | ▓▓▓
                game | 0.1253  | ▓▓▓
              player | 0.1248  | ▓▓▓
             amazing | 0.1221  | ▓▓▓
                  in | 0.1195  | ▓▓▓
                goal | 0.1148  | ▓▓▓

문장: 'neural network model achieved best performance results'  →  예측: 기술
                  단어 |     가중치 | 시각화
  --------------------------------------------------
            achieved | 0.1480  | ▓▓▓▓
              neural | 0.1469  | ▓▓▓▓
             results | 0.1429  | ▓▓▓▓
         performance | 0.1426  | ▓▓▓▓
                best | 0.1411  | ▓▓▓▓
               model | 0.1400  | ▓▓▓▓
             network | 0.1384  | ▓▓▓▓

문장: 'chef cooked delicious pasta with fresh ingredients'  →  예측: 음식
                  단어 |     가중치 | 시각화
  -----

## 7️⃣ 과제 — Ablation Study

다양한 하이퍼파라미터 설정의 성능을 체계적으로 비교합니다.

| 실험 변수 | 설정 값 |
|----------|--------|
| Pooling 전략 | `last`, `max`, `attention` |
| num_layers | `1`, `2` |
| hidden_dim | `32`, `64`, `128` |
| dropout | `0.0`, `0.3`, `0.5` |

In [8]:
experiments = [
    # (이름, 설정)
    ('Pooling=last,   L=2, H=64, drop=0.3', dict(hidden_dim=64,  num_layers=2, dropout=0.3, pooling='last')),
    ('Pooling=max,    L=2, H=64, drop=0.3', dict(hidden_dim=64,  num_layers=2, dropout=0.3, pooling='max')),
    ('Pooling=attn,   L=2, H=64, drop=0.3', dict(hidden_dim=64,  num_layers=2, dropout=0.3, pooling='attention')),
    ('Pooling=attn,   L=1, H=64, drop=0.3', dict(hidden_dim=64,  num_layers=1, dropout=0.3, pooling='attention')),
    ('Pooling=attn,   L=2, H=32, drop=0.3', dict(hidden_dim=32,  num_layers=2, dropout=0.3, pooling='attention')),
    ('Pooling=attn,   L=2, H=128,drop=0.3', dict(hidden_dim=128, num_layers=2, dropout=0.3, pooling='attention')),
    ('Pooling=attn,   L=2, H=64, drop=0.0', dict(hidden_dim=64,  num_layers=2, dropout=0.0, pooling='attention')),
    ('Pooling=attn,   L=2, H=64, drop=0.5', dict(hidden_dim=64,  num_layers=2, dropout=0.5, pooling='attention')),
]

print('=' * 65)
print('[Ablation Study]')
print('=' * 65)
ablation_results = []
for name, cfg in experiments:
    acc, _ = run_experiment(cfg, train_loader, test_loader, epochs=40, verbose=False)
    ablation_results.append((name, acc))
    bar = '█' * int(acc * 20)
    print(f'{name:<42} | {acc*100:5.1f}% | {bar}')

best_name, best_acc = max(ablation_results, key=lambda x: x[1])
print(f'\n★ 최고 설정: {best_name}  ({best_acc*100:.1f}%)')

[Ablation Study]
Pooling=last,   L=2, H=64, drop=0.3        |  25.0% | █████
Pooling=max,    L=2, H=64, drop=0.3        |  25.0% | █████
Pooling=attn,   L=2, H=64, drop=0.3        |  25.0% | █████
Pooling=attn,   L=1, H=64, drop=0.3        |  62.5% | ████████████
Pooling=attn,   L=2, H=32, drop=0.3        |  37.5% | ███████
Pooling=attn,   L=2, H=128,drop=0.3        |  37.5% | ███████
Pooling=attn,   L=2, H=64, drop=0.0        |  25.0% | █████
Pooling=attn,   L=2, H=64, drop=0.5        |  25.0% | █████

★ 최고 설정: Pooling=attn,   L=1, H=64, drop=0.3  (62.5%)


## 8️⃣ 새 문장 예측

In [9]:
best_model.eval()
new_sentences = [
    'olympic sprinter won gold medal breaking world record',
    'transformer model revolutionized natural language processing',
    'italian pizza is baked in wood fired stone oven',
    'backpacking through europe visiting museums and castles',
    'the goalkeeper saved penalty kick in crucial match',
    'python programming language is popular for data science',
    'sushi restaurant in tokyo offers fresh omakase experience',
    'gpu accelerates matrix multiplication for deep learning',
]

print('[새 문장 주제 분류]\n')
with torch.no_grad():
    for sent in new_sentences:
        ids, length = vocab.encode(sent, MAX_LEN)
        x       = torch.tensor([ids],    device=device)
        lengths = torch.tensor([length], device=device)
        probs   = torch.softmax(best_model(x, lengths), dim=1)[0]
        pred    = LABEL_NAMES[probs.argmax().item()]
        conf    = probs.max().item()
        print(f"{sent}")
        for i, (name, p) in enumerate(zip(LABEL_NAMES.values(), probs.tolist())):
            bar = '█' * int(p * 20)
            mark = ' ← 예측' if i == probs.argmax().item() else ''
            print(f"  {name}: {p:.3f} {bar}{mark}")
        print()

[새 문장 주제 분류]

olympic sprinter won gold medal breaking world record
  스포츠: 0.051 █
  기술: 0.010 
  음식: 0.934 ██████████████████ ← 예측
  여행: 0.005 

transformer model revolutionized natural language processing
  스포츠: 0.003 
  기술: 0.001 
  음식: 0.002 
  여행: 0.994 ███████████████████ ← 예측

italian pizza is baked in wood fired stone oven
  스포츠: 0.022 
  기술: 0.021 
  음식: 0.396 ███████
  여행: 0.560 ███████████ ← 예측

backpacking through europe visiting museums and castles
  스포츠: 0.002 
  기술: 0.001 
  음식: 0.121 ██
  여행: 0.876 █████████████████ ← 예측

the goalkeeper saved penalty kick in crucial match
  스포츠: 0.058 █
  기술: 0.033 
  음식: 0.164 ███
  여행: 0.745 ██████████████ ← 예측

python programming language is popular for data science
  스포츠: 0.004 
  기술: 0.026 
  음식: 0.949 ██████████████████ ← 예측
  여행: 0.022 

sushi restaurant in tokyo offers fresh omakase experience
  스포츠: 0.002 
  기술: 0.001 
  음식: 0.752 ███████████████ ← 예측
  여행: 0.245 ████

gpu accelerates matrix multiplication for deep learning
  스